In [1]:
import os, gdown, gc, random
from colorama import Fore
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer, word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer, LancasterStemmer, SnowballStemmer
import contractions
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity


nltk.download('stopwords')
nltk.download('wordnet')
nltk.download("punkt")
nltk.download('omw-1.4')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saivarshith\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\saivarshith\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\saivarshith\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\saivarshith\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\saivarshith\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
# DataSet main Source
# To download - https://www.kaggle.com/allen-institute-for-ai/CORD-19-research-challenge

# globals
# https://drive.google.com/uc?id=1VkNpuudQnlj7g5uUCNPJ4MKxFdDdh7bZ
DS_NAME = "metadata.csv"
BATCH_SIZE=64
dataSetDriveID = "1VkNpuudQnlj7g5uUCNPJ4MKxFdDdh7bZ"

In [3]:
# function to download the dataset from the url
def download_dataset_from_url(url, file_path):
    try:
        gdown.download(url, file_path, quiet=False)
        print(f"Successfully downloaded to {file_path}")
    except Exception as e:
        print(f"Error downloading file: {e}")

def download_and_prep_dataset(FILE_ID, downloadLoc="dataSet", dataSetName=""):
    # Download the dataset
    url = f'https://drive.google.com/uc?id={FILE_ID}'
    currentPath = os.getcwd()
    print("CWD: ", currentPath)
    dataSetPath = os.path.abspath(os.path.join(currentPath, downloadLoc + "/"))

    # create directory if not exists
    if not os.path.exists(dataSetPath):
        os.mkdir(dataSetPath)

    fileFullPath = os.path.join(dataSetPath)
    # download the dataset
    if not os.path.exists(fileFullPath):
        download_dataset_from_url(url=url, file_path=dataSetPath)

    # Load the dataset
    return fileFullPath

In [7]:
# datasetLoc = download_and_prep_dataset(FILE_ID=dataSetDriveID)
currDir = os.getcwd()
datasetLoc = os.path.join(currDir, "dataSet", DS_NAME)
coronaDS = pd.read_csv(datasetLoc)

columnsToKeep = ['title', 'abstract', 'authors', 'publish_time', 'journal', 'url']
coronaDS = coronaDS[columnsToKeep]
coronaDS.head(3)

C:\Users\saivarshith\AppData\Local\Temp\ipykernel_12760\2938802104.py:4: DtypeWarning: Columns (1,4,5,6,13,14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  coronaDS = pd.read_csv(datasetLoc)


,title,abstract,authors,publish_time,journal,url
0,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...,"Madani, Tariq A; Al-Ghamdi, Aisha A",2001-07-04,BMC Infect Dis,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3...
1,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...,"Vliet, Albert van der; Eiserich, Jason P; Cros...",2000-08-15,Respir Res,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...
2,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...,"Crouch, Erika C",2000-08-25,Respir Res,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5...


### Initial Data Exploration

In [16]:
print(Fore.YELLOW)
print(coronaDS.info())
print(Fore.RESET)

print(Fore.LIGHTGREEN_EX, "As the search engine's task is to compare the query with the Journal Abstract and Title, it is safe to drop the documents/records which have missing values in these 2 columns")
print(Fore.RESET)

coronaDS.dropna(subset=['title', 'abstract'], inplace=True)
coronaDS.reset_index(drop=True, inplace=True)
coronaDS['title'] = coronaDS['title'].astype(str)
coronaDS['abstract'] = coronaDS['abstract'].astype(str)
coronaDS['authors'] = coronaDS['authors'].astype(str)
coronaDS['publish_time'] = coronaDS['publish_time'].astype(str)
coronaDS['journal'] = coronaDS['journal'].astype(str)
coronaDS['url'] = coronaDS['url'].astype(str)

print(Fore.YELLOW)
print(coronaDS.info())
print(Fore.RESET)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 821005 entries, 0 to 821004
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   title         821005 non-null  object
 1   abstract      821005 non-null  object
 2   authors       821005 non-null  object
 3   publish_time  821005 non-null  object
 4   journal       821005 non-null  object
 5   url           821005 non-null  object
dtypes: object(6)
memory usage: 37.6+ MB
None

 As the search engine's task is to compare the query with the Journal Abstract and Title, it is safe to drop the documents/records which have missing values in these 2 columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 821005 entries, 0 to 821004
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   title         821005 non-null  object
 1   abstract      821005 non-null  object
 2   authors       821005 non-null  ob

After clean-up we can see that there are no missing Values and the Dataset is clean to work with